In [16]:
import os
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

INTERIM_DIR = os.path.join("..", "data", "interim")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [17]:
def load_fact_table() -> pd.DataFrame:
    parquet_path = os.path.join(INTERIM_DIR, "fact_validations.parquet")
    pkl_path = os.path.join(INTERIM_DIR, "fact_validations.pkl")
    if os.path.exists(parquet_path):
        try:
            return pd.read_parquet(parquet_path)
        except ImportError:
            pass
    return pd.read_pickle(pkl_path)


master_table = load_fact_table()
master_table["day_type"] = np.where(master_table["is_weekend"], "weekend", "weekday")
print("master_table:", master_table.shape)

master_table: (67407, 36)


## Shared helpers

In [18]:
manifest_rows = []


def assert_no_nulls_in_keys(df: pd.DataFrame, key_cols: list, table_name: str) -> None:
    null_counts = df[key_cols].isna().sum()
    bad = null_counts[null_counts > 0]
    assert bad.empty, f"{table_name}: nulls found in key columns:\n{bad}"


def save_table(df: pd.DataFrame, name: str, key_cols: list) -> None:
    assert_no_nulls_in_keys(df, key_cols, name)

    other_nulls = df.drop(columns=key_cols).isna().sum()
    other_nulls = other_nulls[other_nulls > 0]
    if not other_nulls.empty:
        print(f"  NOTE ({name}): nulls in non-key columns (not blocked, but worth a look):")
        print(f"  {other_nulls.to_dict()}")

    path = os.path.join(PROCESSED_DIR, f"{name}.csv")
    df.to_csv(path, index=False, encoding="utf-8")

    manifest_rows.append({
        "file": f"{name}.csv",
        "rows": len(df),
        "columns": df.shape[1],
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    })
    print(f"saved {name}.csv ({len(df):,} rows, {df.shape[1]} columns)")

### Shared vehicle trip table 

In [19]:
trip = master_table.groupby(
    ["vehicle_id", "route_id", "mode", "time_period", "year_month", "date", "hour"],
    as_index=False,
).agg(
    total_passengers=("passenger_count", "sum"),
    capacity=("capacity", "first"),
)
trip["load_factor"] = trip["total_passengers"] / trip["capacity"]
print(f"{len(trip):,} vehicle-trip rows")


67,108 vehicle-trip rows


### AGG_ridership_by_month --- grain:  year_month

In [20]:
monthly = master_table.groupby("year_month", as_index=False).agg(
    total_boardings=("passenger_count", "sum"),
    total_revenue=("revenue", "sum"),
    unique_cards=("card_id", lambda s: s[s != "SINGLE_RIDE"].nunique()),
)

days_per_month = master_table.groupby("year_month")["date"].nunique()
monthly["avg_daily_boardings"] = monthly["total_boardings"] / monthly["year_month"].map(days_per_month)

monthly = monthly.sort_values("year_month").reset_index(drop=True)
monthly["mom_growth_pct"] = (monthly["total_boardings"].pct_change() * 100).round(4)

monthly["total_revenue"] = monthly["total_revenue"].round(2)
monthly["avg_daily_boardings"] = monthly["avg_daily_boardings"].round(2)

save_table(monthly, "agg_ridership_by_month", key_cols=["year_month"])
monthly

  NOTE (agg_ridership_by_month): nulls in non-key columns (not blocked, but worth a look):
  {'mom_growth_pct': 1}
saved agg_ridership_by_month.csv (6 rows, 6 columns)


,year_month,total_boardings,total_revenue,unique_cards,avg_daily_boardings,mom_growth_pct
0,2024-01,12405.0,22784.39,8323,400.16,NaN
1,2024-02,11598.0,21405.37,7926,399.93,-6.5054
2,2024-03,12379.0,22674.18,8294,399.32,6.7339
3,2024-04,11835.0,21742.23,8118,394.50,-4.3945
4,2024-05,12552.0,22860.17,8366,404.90,6.0583
5,2024-06,11872.0,21856.48,8047,395.73,-5.4175


### AGG_ridership_by_hour ---grain: hour, day_type

In [21]:
hourly = master_table.groupby(["hour", "day_type"], as_index=False).agg(
    total_boardings=("passenger_count", "sum"),
)

days_by_type = master_table.groupby("day_type")["date"].nunique()
hourly["avg_boardings_per_day"] = hourly["total_boardings"] / hourly["day_type"].map(days_by_type)

hourly["daily_total"] = hourly.groupby("day_type")["total_boardings"].transform("sum")
hourly["pct_of_daily_total"] = (hourly["total_boardings"] / hourly["daily_total"] * 100).round(4)
hourly = hourly.drop(columns="daily_total")

hourly["avg_boardings_per_day"] = hourly["avg_boardings_per_day"].round(2)
hourly = hourly.sort_values(["day_type", "hour"]).reset_index(drop=True)

save_table(hourly, "agg_ridership_by_hour", key_cols=["hour", "day_type"])
hourly.head(10)

saved agg_ridership_by_hour.csv (48 rows, 5 columns)


,hour,day_type,total_boardings,avg_boardings_per_day,pct_of_daily_total
0,0,weekday,125.0,0.96,0.2411
1,1,weekday,98.0,0.75,0.1890
2,2,weekday,141.0,1.08,0.2719
3,3,weekday,105.0,0.81,0.2025
4,4,weekday,151.0,1.16,0.2912
5,5,weekday,318.0,2.45,0.6133
6,6,weekday,1333.0,10.25,2.5707
7,7,weekday,4122.0,31.71,7.9494
8,8,weekday,6344.0,48.80,12.2346
9,9,weekday,4692.0,36.09,9.0487


### AGG_route_performance --- grain: route_id, route_name, mode, district, year_month

In [22]:
route_perf = master_table.groupby(
    ["route_id", "route_name", "mode", "route_district", "year_month"], as_index=False
).agg(
    total_boardings=("passenger_count", "sum"),
    total_revenue=("revenue", "sum"),
    route_length_km=("route_length_km", "first"),
)
route_perf = route_perf.rename(columns={"route_district": "district"})
route_perf["boardings_per_km"] = route_perf["total_boardings"] / route_perf["route_length_km"]
route_perf = route_perf.drop(columns="route_length_km")

avg_lf_route = trip.groupby(["route_id", "year_month"], as_index=False).agg(
    avg_load_factor=("load_factor", "mean")
)
route_perf = route_perf.merge(avg_lf_route, on=["route_id", "year_month"], how="left")

route_perf["total_revenue"] = route_perf["total_revenue"].round(2)
route_perf["boardings_per_km"] = route_perf["boardings_per_km"].round(2)
route_perf["avg_load_factor"] = route_perf["avg_load_factor"].round(4)

route_perf = route_perf.sort_values(
    ["year_month", "total_boardings"], ascending=[True, False]
).reset_index(drop=True)

save_table(route_perf, "agg_route_performance",
           key_cols=["route_id", "route_name", "mode", "district", "year_month"])
route_perf.head(10)


saved agg_route_performance.csv (360 rows, 9 columns)


,route_id,route_name,mode,district,year_month,total_boardings,total_revenue,boardings_per_km,avg_load_factor
0,R032,Thomas Dam Line,Metro,Centro,2024-01,250.0,419.61,10.64,0.0019
1,R048,Eric Track Line,Bus,Eastfield,2024-01,247.0,456.70,24.22,0.0138
2,R039,Carlson Mountain Line,Bus,Centro,2024-01,246.0,434.08,58.57,0.014
3,R051,Daniel Lake Line,Bus,Centro,2024-01,242.0,445.65,49.39,0.014
4,R031,Wilkerson Row Line,Tram,Centro,2024-01,235.0,434.16,13.91,0.0055
5,R021,Martin Knoll Line,Tram,Westend,2024-01,230.0,436.57,9.75,0.0057
6,R026,Gabrielle Ville Line,Bus,Riverside,2024-01,228.0,396.85,9.34,0.0138
7,R041,Erin Crescent Line,Bus,Northgate,2024-01,225.0,430.49,26.47,0.0139
8,R010,Moore Track Line,Bus,Hillcrest,2024-01,224.0,416.50,14.27,0.0135
9,R020,Chavez Village Line,Bus,Old Town,2024-01,224.0,417.70,22.40,0.0134


### AGG_mode_summary --grain :mode, year_month

In [23]:
mode_summary = master_table.groupby(["mode", "year_month"], as_index=False).agg(
    total_boardings=("passenger_count", "sum"),
    total_revenue=("revenue", "sum"),
    route_count=("route_id", "nunique"),
)
mode_summary["boardings_per_route"] = mode_summary["total_boardings"] / mode_summary["route_count"]

mode_summary = mode_summary.sort_values(["mode", "year_month"]).reset_index(drop=True)
mode_summary["mom_growth_pct"] = (
    mode_summary.groupby("mode")["total_boardings"].transform(lambda s: s.pct_change()) * 100
).round(4)

mode_summary["total_revenue"] = mode_summary["total_revenue"].round(2)
mode_summary["boardings_per_route"] = mode_summary["boardings_per_route"].round(2)

save_table(mode_summary, "agg_mode_summary", key_cols=["mode", "year_month"])
mode_summary

  NOTE (agg_mode_summary): nulls in non-key columns (not blocked, but worth a look):
  {'mom_growth_pct': 3}
saved agg_mode_summary.csv (18 rows, 7 columns)


,mode,year_month,total_boardings,total_revenue,route_count,boardings_per_route,mom_growth_pct
0,Bus,2024-01,7141.0,13171.17,34,210.03,NaN
1,Bus,2024-02,6550.0,12088.91,34,192.65,-8.2762
2,Bus,2024-03,6963.0,12728.86,34,204.79,6.3053
3,Bus,2024-04,6730.0,12360.48,34,197.94,-3.3463
4,Bus,2024-05,7171.0,13082.33,34,210.91,6.5527
5,Bus,2024-06,6568.0,12091.79,34,193.18,-8.4089
6,Metro,2024-01,1200.0,2165.75,6,200.00,NaN
7,Metro,2024-02,1068.0,2021.18,6,178.00,-11.0000
8,Metro,2024-03,1218.0,2184.74,6,203.00,14.0449
9,Metro,2024-04,1124.0,2043.15,6,187.33,-7.7176


### AGG_fare_type_mix --- grain: fare_type, year_month

In [24]:
fare_mix = master_table.groupby(["fare_type", "year_month"], as_index=False).agg(
    boardings=("passenger_count", "sum"),
    revenue=("revenue", "sum"),
)

fare_mix["monthly_total"] = fare_mix.groupby("year_month")["boardings"].transform("sum")
fare_mix["pct_of_monthly_boardings"] = (fare_mix["boardings"] / fare_mix["monthly_total"] * 100).round(4)
fare_mix = fare_mix.drop(columns="monthly_total")

fare_mix["revenue"] = fare_mix["revenue"].round(2)

fare_mix = fare_mix.sort_values(
    ["year_month", "boardings"], ascending=[True, False]
).reset_index(drop=True)

save_table(fare_mix, "agg_fare_type_mix", key_cols=["fare_type", "year_month"])
fare_mix.head(10)


saved agg_fare_type_mix.csv (30 rows, 5 columns)


,fare_type,year_month,boardings,revenue,pct_of_monthly_boardings
0,Adult,2024-01,6875.0,17179.27,55.4212
1,Student,2024-01,1856.0,2788.88,14.9617
2,Senior,2024-01,1478.0,1768.20,11.9146
3,Pass Holder,2024-01,1216.0,70.80,9.8025
4,Child,2024-01,980.0,977.24,7.9000
5,Adult,2024-02,6473.0,16192.88,55.8113
6,Student,2024-02,1754.0,2628.32,15.1233
7,Senior,2024-02,1349.0,1613.40,11.6313
8,Pass Holder,2024-02,1114.0,62.50,9.6051
9,Child,2024-02,908.0,908.27,7.8289


### AGG_stop_activity ---grain: stop_id,stop_name, district, zone

In [25]:
stop_activity = master_table.groupby(
    ["stop_id", "stop_name", "stop_district", "zone", "has_shelter"], as_index=False
).agg(
    total_boardings=("passenger_count", "sum"),
)
stop_activity = stop_activity.rename(columns={"stop_district": "district"})

stop_activity["rank_within_district"] = (
    stop_activity.groupby("district")["total_boardings"]
    .rank(method="min", ascending=False)
    .astype(int)
)

stop_activity = stop_activity.sort_values(
    ["district", "rank_within_district"]
).reset_index(drop=True)

save_table(stop_activity, "agg_stop_activity", key_cols=["stop_id", "stop_name", "district", "zone"])
stop_activity.head(10)


saved agg_stop_activity.csv (400 rows, 7 columns)


,stop_id,stop_name,district,zone,has_shelter,total_boardings,rank_within_district
0,S0047,Joseph Coves Stop,Centro,Zone 3,False,206.0,1
1,S0323,Brandon Camp Stop,Centro,Zone 1,False,201.0,2
2,S0014,James Plain Stop,Centro,Zone 3,False,200.0,3
3,S0394,Tucker Mills Stop,Centro,Zone 2,False,199.0,4
4,S0103,Jennifer Lights Stop,Centro,Zone 1,False,196.0,5
5,S0314,Davis Locks Stop,Centro,Zone 2,True,196.0,5
6,S0021,Lewis Locks Stop,Centro,Zone 3,False,195.0,7
7,S0186,James Streets Stop,Centro,Zone 2,False,195.0,7
8,S0192,Terry Crossroad Stop,Centro,Zone 3,False,194.0,9
9,S0350,Thomas Tunnel Stop,Centro,Zone 1,False,193.0,10


### 7. AGG_vehicle_utilization ---grain: vehicle_id, vehicle_type, depot, vehicle_age_band

In [26]:
vehicle_util = master_table.groupby(
    ["vehicle_id", "vehicle_type", "depot", "vehicle_age_band"], as_index=False
).agg(
    total_boardings=("passenger_count", "sum"),
    active_days=("date", "nunique"),
)

avg_lf_vehicle = trip.groupby("vehicle_id", as_index=False).agg(avg_load_factor=("load_factor", "mean"))
vehicle_util = vehicle_util.merge(avg_lf_vehicle, on="vehicle_id", how="left")

vehicle_util["avg_load_factor"] = vehicle_util["avg_load_factor"].round(4)

vehicle_util = vehicle_util.sort_values("avg_load_factor", ascending=True).reset_index(drop=True)

save_table(vehicle_util, "agg_vehicle_utilisation",
           key_cols=["vehicle_id", "vehicle_type", "depot", "vehicle_age_band"])
vehicle_util.head(10)


saved agg_vehicle_utilisation.csv (180 rows, 7 columns)


,vehicle_id,vehicle_type,depot,vehicle_age_band,total_boardings,active_days,avg_load_factor
0,V0016,Metro Set,Central Depot,6-10,327.0,144,0.0016
1,V0030,Metro Set,East Depot,11-20,358.0,149,0.0016
2,V0113,Metro Set,North Depot,20+,348.0,160,0.0016
3,V0078,Metro Set,Central Depot,0-5,319.0,145,0.0016
4,V0024,Metro Set,Riverside Depot,20+,375.0,153,0.0017
5,V0051,Metro Set,Riverside Depot,20+,313.0,142,0.0017
6,V0164,Metro Set,Riverside Depot,0-5,324.0,139,0.0017
7,V0138,Metro Set,East Depot,20+,320.0,151,0.0017
8,V0067,Metro Set,Central Depot,20+,352.0,151,0.0018
9,V0073,Metro Set,Central Depot,6-10,350.0,156,0.0018


### 8. AGG_peak_analysis -- grain: time_period, mode

In [27]:
print("Values in master_table:")
print(master_table["time_period"].unique())
print("\n My expected list:")

Values in master_table:
<ArrowStringArray>
['AM peak (07-09)', 'Midday (10-15)', 'PM Peak(16-18)', 'Evening(19-23)', 'Early (04-06)', 'Night(00-03)']
Length: 6, dtype: str

 My expected list:


In [28]:
peak = master_table.groupby(["time_period", "mode"], as_index=False).agg(
    total_boardings=("passenger_count", "sum"),
)

avg_lf_peak = trip.groupby(["time_period", "mode"], as_index=False).agg(
    avg_load_factor=("load_factor", "mean")
)
peak = peak.merge(avg_lf_peak, on=["time_period", "mode"], how="left")

peak["mode_total"] = peak.groupby("mode")["total_boardings"].transform("sum")
peak["pct_of_mode_total"] = (peak["total_boardings"] / peak["mode_total"] * 100).round(4)
peak = peak.drop(columns="mode_total")

peak["avg_load_factor"] = peak["avg_load_factor"].round(4)

TIME_PERIOD_ORDER = ['AM peak (07-09)', 'Midday (10-15)', 'PM Peak(16-18)', 'Evening(19-23)', 'Early (04-06)', 'Night(00-03)']
peak["time_period"] = pd.Categorical(peak["time_period"], categories=TIME_PERIOD_ORDER, ordered=True)
peak = peak.sort_values(["mode", "time_period"]).reset_index(drop=True)
peak["time_period"] = peak["time_period"].astype(str)  # plain string for CSV export

save_table(peak, "agg_peak_analysis", key_cols=["time_period", "mode"])
peak


saved agg_peak_analysis.csv (18 rows, 5 columns)


,time_period,mode,total_boardings,avg_load_factor,pct_of_mode_total
0,AM peak (07-09),Bus,12074.0,0.0135,29.3607
1,Midday (10-15),Bus,10173.0,0.0135,24.7380
2,PM Peak(16-18),Bus,13028.0,0.0134,31.6806
3,Evening(19-23),Bus,4088.0,0.0135,9.9409
4,Early (04-06),Bus,1398.0,0.0133,3.3996
5,Night(00-03),Bus,362.0,0.0126,0.8803
6,AM peak (07-09),Metro,2081.0,0.0018,29.5471
7,Midday (10-15),Metro,1749.0,0.0018,24.8332
8,PM Peak(16-18),Metro,2228.0,0.0018,31.6342
9,Evening(19-23),Metro,696.0,0.0018,9.8822


### 9. AGG_transfer_behavior --grain: year_month,mode

In [29]:
transfer = master_table.groupby(["year_month", "mode"], as_index=False).agg(
    transfer_count=("transfer_flag", "sum"),
    total_validations=("validation_id", "count"),
)
transfer["transfer_rate"] = (transfer["transfer_count"] / transfer["total_validations"]).round(4)
transfer = transfer.drop(columns="total_validations")

transfer = transfer.sort_values(["year_month", "mode"]).reset_index(drop=True)

save_table(transfer, "agg_transfer_behaviour", key_cols=["year_month", "mode"])
transfer


saved agg_transfer_behaviour.csv (18 rows, 4 columns)


,year_month,mode,transfer_count,transfer_rate
0,2024-01,Bus,2491,0.3774
1,2024-01,Metro,418,0.3776
2,2024-01,Tram,1429,0.3749
3,2024-02,Bus,2252,0.3698
4,2024-02,Metro,341,0.34
5,2024-02,Tram,1405,0.3841
6,2024-03,Bus,2481,0.3856
7,2024-03,Metro,426,0.3855
8,2024-03,Tram,1482,0.3799
9,2024-04,Bus,2387,0.3802


### Manifest (one raw per output file: name, row count, column count, and when it was generated)

In [30]:
manifest = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(PROCESSED_DIR, "_manifest.csv")
manifest.to_csv(manifest_path, index=False, encoding="utf-8")
print(f"saved {manifest_path} ({len(manifest)} entries)")
manifest

saved ..\data\processed\_manifest.csv (9 entries)


,file,rows,columns,generated_at
0,agg_ridership_by_month.csv,6,6,2026-09-20T09:08:17
1,agg_ridership_by_hour.csv,48,5,2026-09-20T09:08:17
2,agg_route_performance.csv,360,9,2026-09-20T09:08:17
3,agg_mode_summary.csv,18,7,2026-09-20T09:08:17
4,agg_fare_type_mix.csv,30,5,2026-09-20T09:08:17
5,agg_stop_activity.csv,400,7,2026-09-20T09:08:17
6,agg_vehicle_utilisation.csv,180,7,2026-09-20T09:08:17
7,agg_peak_analysis.csv,18,5,2026-09-20T09:08:17
8,agg_transfer_behaviour.csv,18,4,2026-09-20T09:08:17
